### 🧠 Core Idea of the Exercise

Below code is simulating multiple personalities (agents) using LLMs.

Each model:

- Has its own identity (system prompt)
- Sees the same conversation history
- Generates the next message

In [ ]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

# For Gemini and Groq, we can use the OpenAI python client
# Because they have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

groq_url = "https://api.groq.com/openai/v1"
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [ ]:
# Let's make a conversation between Groq and Gemini.
# We're using free versions of models so the costs is zero

gemini_model = "gemini-2.5-flash-lite"

groq_model1 = "llama-3.1-8b-instant"

groq_model2 = "llama-3.1-8b-instant"

gemini_system = "You are Alex, a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

groq_system1 = "You are Blake, a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

groq_system2 = "You are Casey, a very sad and empathetic chatbot. You try to add sadness \
and empathy to the conversation as if these small moments of does not matter in the \
grand scheme of things."

conversation = [("Alex", "Hi there"), ("Blake", "Hi"), ("Casey", "Hello")]

In [ ]:
def call_gemini():
    messages = [{"role": "system", "content": gemini_system}]
    convo_text = ""
    for speaker, msg in conversation:
        convo_text += f"{speaker}: {msg}\n"
    
    user_prompt = f"You are Alex, in conversation with Blake and Casey. \
    The conversation so far is as follows: {convo_text} \
    Now with this, respond with what you would like to say next, as Alex."
    
    messages.append({"role": "user", "content": user_prompt})
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
def call_groq1():
    messages = [{"role": "system", "content": groq_system1}]
    convo_text = ""
    for speaker, msg in conversation:
        convo_text += f"{speaker}: {msg}\n"
        
    user_prompt = f"You are Blake, in conversation with Alex and Casey. \
    The conversation so far is as follows: {convo_text} \
    Now with this, respond with what you would like to say next, as Blake."
    
    messages.append({"role": "user", "content": user_prompt})
    response = groq.chat.completions.create(model=groq_model1, messages=messages)
    return response.choices[0].message.content

In [ ]:
def call_groq2():
    messages = [{"role": "system", "content": groq_system2}]
    convo_text = ""
    for speaker, msg in conversation:
        convo_text += f"{speaker}: {msg}\n"
    
    user_prompt = f"You are Casey, in conversation with Alex and Blake. \
    The conversation so far is as follows: {convo_text} \
    Now with this, respond with what you would like to say next, as Casey."
    
    messages.append({"role": "user", "content": user_prompt})
    response = groq.chat.completions.create(model=groq_model2, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_groq1()

In [ ]:
call_gemini()

In [ ]:
call_groq2()

In [ ]:
for speaker, msg in conversation:
    display(Markdown(f"### {speaker}:\n{msg}\n"))
    
for i in range(1):
    gemini_next = call_gemini()
    display(Markdown(f"### Alex:\n{gemini_next}\n"))
    conversation.append(("Alex", gemini_next))

    groq1_next = call_groq1()
    display(Markdown(f"### Blake:\n{groq1_next}\n"))
    conversation.append(("Blake", groq1_next))

    groq2_next = call_groq2()
    display(Markdown(f"### Casey:\n{groq2_next}\n"))
    conversation.append(("Casey", groq2_next))

### The same functionality can be implemented using a single function with arguments, replacing all three of the above functions.

In [ ]:
def call_agent(client, run_Model, system_prompt, conversation, speaker_name):
    messages = [{"role": "system", "content": system_prompt}]
    convo_text = ""
    for speaker, msg in conversation:
        convo_text += f"{speaker}: {msg}\n"
        
    user_prompt = f"You are {speaker_name}, in conversation with two other participants. \
    The conversation so far is as follows: {convo_text} \
    Now with this, respond with what you would like to say next, as {speaker_name}."
    
    messages.append({"role": "user", "content": user_prompt})
    response = client.chat.completions.create(model=run_Model, messages=messages)
    return response.choices[0].message.content

In [ ]:
for speaker, msg in conversation:
    display(Markdown(f"### {speaker}:\n{msg}\n"))
    
for i in range(1):
    alex = call_agent(gemini, gemini_model, gemini_system, conversation, "Alex")
    display(Markdown(f"### Alex:\n{alex}\n"))
    conversation.append(("Alex", alex))

    blake = call_agent(groq, groq_model1, groq_system1, conversation, "Blake")
    display(Markdown(f"### Blake:\n{blake}\n"))
    conversation.append(("Blake", blake))

    casey = call_agent(groq, groq_model2, groq_system2, conversation, "Casey")
    display(Markdown(f"### Casey:\n{casey}\n"))
    conversation.append(("Casey", casey))